In [11]:
import requests
from bs4 import BeautifulSoup
import os

# Dati di accesso quando necessario il login.
username = 'Martinelli'
password = 'RA5fje2xGk9CdPUFtxSwt^#Y'
login_url = 'https://www.martiniprofessional.it/wp-login.php'

## Rimuove il prefisso base dall'URL
base_url = 'https://www.martiniprofessional.it/'
target_url = 'https://www.martiniprofessional.it/zh-hans/'

## link che devono essere ricercati
start_with = 'https://www.martiniprofessional.it/zh-hans/'

pages = []

def login():
    # Effettua il login
    session = requests.Session()
    payload = {
        'log': username,
        'pwd': password,
        'wp-submit': 'Log+In',
        'redirect_to': target_url,
        'testcookie': '1'
    }
    response = session.post(login_url, data=payload)

    # Verifica che il login sia avvenuto con successo
    if response.status_code == 200:
        print("Login effettuato con successo.")
    else:
        print("Errore nel login.")
        exit(1)
    return session

session = login()

# Funzione per ottenere tutti i link da una pagina
def get_all_links_from_page(url):
    global session
    response = session.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    links = [link.get('href') for link in soup.find_all('a', href=True)]
    return links

# Funzione principale per ottenere tutti i link del sito
def get_all_site_links(base_url):
    global start_with
    visited_urls = set()
    links_to_visit = [base_url]

    while links_to_visit:
        current_url = links_to_visit.pop(0)
        if current_url not in visited_urls:
            visited_urls.add(current_url)
            links = get_all_links_from_page(current_url)
            for link in links:
                if link.startswith(start_with) and link not in visited_urls:
                    links_to_visit.append(link)

    return visited_urls

def download_file(pages):
    global session
    # Scarica tutte le pagine /zh-hant
    for page in pages:
        url = base_url + page if base_url not in page else page
        try:
            response = session.get(url)
            print("sessione")
            if response.status_code == 200:
                print("200")
                page = page.replace(base_url, '/') 
                filename = os.path.join('output', page.replace('/', '_') + '.html')
                
                print(filename)
                with open(filename, 'w', encoding='utf-8') as file:
                    print("scaricare")
                    file.write(response.text)
                print(f"Scaricato: {filename}")
            else:
                print(f"Errore nel caricamento della pagina: {url}")
        except Exception as e:
            print(f"Errore durante il download della pagina: {url} - {str(e)}")

    print("Download completato.")


def download_site():
    global pages, target_url
    if len(pages) == 0:
        print("@@@@@@@@@@@@@@@@")
        # Ottieni tutti i link del sito
        pages = get_all_site_links(target_url)
        print("pages")
        pages = [page[len(base_url):] for page in pages]
        print(pages)
    print(pages)
    
    download_file(pages)

download_site()


Login effettuato con successo.
@@@@@@@@@@@@@@@@


TypeError: get_all_links_from_page() takes 1 positional argument but 2 were given